# GDBN v4: Point-in-Time, Walk-Forward Rankings

The fixed 30-stock universe was chosen with **2026 hindsight** (survivorship bias). Here the universe is **point-in-time**:

- **Universe:** on the first trading day of each quarter, the **5 largest S&P 500 members in each sector** (by market cap on the previous day), about 50–55 stocks. Companies that later failed (e.g., AIG in 2008) are included while they were large.
- **Walk-forward:** each quarter, the network is **re-learned on only the past 3 years**, then used to rank stocks every day of the next quarter. Every prediction is out of sample.
- **Nodes:** for each stock, yesterday's **excess return** (vs the universe average), **news sentiment**, and **news attention**, plus 5 market/commodity ETFs (SPY, USO, GLD, DBC, TLT). Targets: each stock's excess return today.
- **Ranking:** each day, buy the top 20% and short the bottom 20% (about 11 stocks each), hold one day.

**Periods** (test years 2023–2025 are **not used** here):

| Period | Role |
|---|---|
| 2006–2020 walk-forward | every prediction is out of sample; used to **choose λ** |
| 2021–2022 | **validation**: a clean check with λ fixed |

**Survivorship test:** the same walk-forward on the **fixed 30 stocks**. The difference between the two is how much hindsight inflated the earlier results.

Data: `build_pit.py` → `data/pit/built/`. Takes a few minutes to run.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.linear_model import lasso_path
from gdbn_utils import DATA, RESULTS

BUILT = DATA / "pit" / "built"
WINDOW = 756                         # training window: 3 years of trading days
TOP_FRAC = 0.20                      # long the top 20%, short the bottom 20%
COST_BPS = 5                         # trading cost per unit traded (bps)
N_RANDOM = 100                       # random portfolios (luck benchmark)
SENT_KIND = "news"                   # "news" = sentiment without price-movement stories, "all" = every story
ALPHAS = np.logspace(-0.7, -3, 20)   # shared λ grid (all variables are z-scores), largest first
INNER = 252                          # λ by MSE: the last year of each training window is the inner validation
TUNE_END, EVAL_END = "2020-12-31", "2022-12-31"   # nothing after EVAL_END is used (test years locked)
ETFS = ["SPY", "USO", "GLD", "DBC", "TLT"]
rng = np.random.default_rng(0)

days = pd.DatetimeIndex(pd.read_csv(BUILT / "trading_days.csv", parse_dates=["date"])["date"])
panel = pd.read_csv(BUILT / "panel.csv.gz", parse_dates=["date"],
                    usecols=["date", "permno", "ret", f"n_{SENT_KIND}", f"sent_{SENT_KIND}"])
RET = panel.pivot(index="date", columns="permno", values="ret").reindex(days)
SENT = panel.pivot(index="date", columns="permno", values=f"sent_{SENT_KIND}").reindex(days)
ATTN = np.log1p(panel.pivot(index="date", columns="permno", values=f"n_{SENT_KIND}")).reindex(days)
ETF = pd.read_csv(BUILT / "etf_returns.csv", index_col="date", parse_dates=True).reindex(days)[ETFS]
universe = pd.read_csv(BUILT / "universe.csv", parse_dates=["rebalance_date"])
sel = universe[universe["selected"]]
del panel

rebal = [pd.Timestamp(r) for r in sorted(sel["rebalance_date"].unique()) if r <= pd.Timestamp(EVAL_END)]
U_PIT = {r: list(g["permno"]) for r, g in sel.groupby("rebalance_date") if r in rebal}
TICKER = sel.sort_values("rebalance_date").drop_duplicates("permno", keep="last").set_index("permno")["ticker"].to_dict()
SECTOR = {(r, p): s for r, p, s in zip(sel["rebalance_date"], sel["permno"], sel["sector_name"])}

fixed = pd.read_csv(DATA / "universe.csv").drop_duplicates("ticker")
fixed = fixed[fixed["rp_entity_id"].notna()]          # the 30 stocks (ETFs have no news entity)
FIXED_PERMNOS = [int(p) for p in fixed["permno"]]
U_FIXED = {r: FIXED_PERMNOS for r in rebal}
TICKER.update(dict(zip(fixed["permno"].astype(int), fixed["ticker"])))
FIXED_SECTOR = dict(zip(fixed["permno"].astype(int), fixed["sector"]))

print(f"{len(rebal)} quarters ({pd.Timestamp(rebal[0]).date()} to {pd.Timestamp(rebal[-1]).date()}); "
      f"point-in-time universe: {sel[sel['rebalance_date'].isin(rebal)]['permno'].nunique()} different stocks over time; "
      f"fixed universe: {len(FIXED_PERMNOS)} stocks")

## 1. Walk-forward: re-learn the network every quarter

For each quarter:
1. **Universe** $U$: the stocks selected on that quarter's first trading day.
2. **Inputs at $t-1$:** each stock's excess return, sentiment, attention, plus the 5 ETFs, z-scored with the **training window's** statistics (missing values, e.g., before a stock existed, are set to the window average).
3. **One lasso per stock** on the past 756 days (the GDBN's arrows into that stock), for every λ on the grid.
4. **Scores** for every day of the quarter: the predicted excess return (dynamic part only, no stock-specific average).
5. **Predicted volatility** for the risk-adjusted score: HAR (yesterday, past week, past month of $\log|r|$), fit on the same window.
6. **λ by MSE (per stock):** fit each stock's lasso on the first 2 years of the window, pick the λ with the lowest error on the **last year** of the window (inner validation), then use that λ with the full 3-year fit. Only past data is used, so there is no look-ahead.

In [ ]:
def fit_quarter(r, nxt, U):
    pos = days.searchsorted(r)
    if pos - WINDOW - 23 < 0:
        return None
    hold = days[pos: days.searchsorted(nxt)]
    hold = hold[hold <= pd.Timestamp(EVAL_END)]
    if len(hold) == 0:
        return None
    win = days[pos - WINDOW: pos]
    span = days[pos - WINDOW - 23: pos + len(hold)]               # extra days for lags and HAR averages

    R = RET.reindex(index=span, columns=U)
    EX = R.sub(R.mean(axis=1), axis=0)                             # excess return vs the universe average
    X = pd.concat([EX.add_suffix("|ret"), SENT.reindex(index=span, columns=U).add_suffix("|sent"),
                   ATTN.reindex(index=span, columns=U).add_suffix("|attn"), ETF.reindex(span)], axis=1).shift(1)
    Xw, Xh = X.loc[win], X.loc[hold]
    mu, sd = Xw.mean(), Xw.std().replace(0, np.nan)
    Xwz = ((Xw - mu) / sd).fillna(0).values
    Xhz = ((Xh - mu) / sd).fillna(0).values
    xm = Xwz.mean(axis=0)

    LV = np.log(R.abs() + 1e-4)                                    # HAR features for predicted volatility
    H1, H5, H22 = LV.shift(1), LV.rolling(5, min_periods=3).mean().shift(1), LV.rolling(22, min_periods=10).mean().shift(1)

    S = np.full((len(ALPHAS), len(hold), len(U)), np.nan)
    W = np.zeros((len(U), X.shape[1], len(ALPHAS)))
    vol = np.full((len(hold), len(U)), np.nan)
    k_mse = np.full(len(U), -1)
    for i, p in enumerate(U):
        y = EX.loc[win, p].values
        ok = ~np.isnan(y)
        if ok.sum() < 252:
            continue
        ym, ys = y[ok].mean(), y[ok].std()
        _, coefs, _ = lasso_path(Xwz[ok] - xm, (y[ok] - ym) / ys, alphas=ALPHAS)
        W[i] = coefs
        S[:, :, i] = ((Xhz - xm) @ coefs).T * ys

        rows_ok = np.where(ok)[0]                                   # λ by MSE: inner validation = last year
        tr_i, va_i = rows_ok[:-INNER], rows_ok[-INNER:]
        if len(tr_i) >= 252:
            yt, yv = (y[tr_i] - ym) / ys, (y[va_i] - ym) / ys
            xmi = Xwz[tr_i].mean(axis=0)
            _, c_in, _ = lasso_path(Xwz[tr_i] - xmi, yt - yt.mean(), alphas=ALPHAS)
            mse = ((yv[:, None] - (yt.mean() + (Xwz[va_i] - xmi) @ c_in)) ** 2).mean(axis=0)
            k_mse[i] = int(np.argmin(mse))

        F = pd.concat([H1[p], H5[p], H22[p]], axis=1)
        fw, lv = F.loc[win], LV.loc[win, p]
        okv = fw.notna().all(axis=1) & lv.notna()
        if okv.sum() > 100:
            A = np.column_stack([np.ones(okv.sum()), fw[okv].values])
            b, *_ = np.linalg.lstsq(A, lv[okv].values, rcond=None)
            fh = F.loc[hold].fillna(fw[okv].mean())
            vol[:, i] = np.exp(b[0] + fh.values @ b[1:])
    return {"r": r, "hold": hold, "U": U, "S": S, "W": W, "cols": list(X.columns), "vol": vol, "k_mse": k_mse}


def walk_forward(U_by_q, label):
    out = []
    for j, r in enumerate(rebal):
        nxt = rebal[j + 1] if j + 1 < len(rebal) else pd.Timestamp(EVAL_END) + pd.Timedelta(days=1)
        res = fit_quarter(pd.Timestamp(r), pd.Timestamp(nxt), U_by_q[r])
        if res is not None:
            out.append(res)
        if (j + 1) % 20 == 0:
            print(f"  {label}: {j + 1} of {len(rebal)} quarters")
    return out


def scores(results, k=None, kind="gdbn"):
    # Assemble a date x permno score table (NaN = not in that day's universe)
    parts = []
    for q in results:
        if kind == "gdbn":
            v = q["S"][k]
        elif kind == "riskadj":
            v = q["S"][k] / q["vol"]
        elif kind == "mse":                                           # each stock's own λ (inner validation)
            v = np.full(q["S"].shape[1:], np.nan)
            for i, kk in enumerate(q["k_mse"]):
                if kk >= 0:
                    v[:, i] = q["S"][kk, :, i]
        parts.append(pd.DataFrame(v, index=q["hold"], columns=q["U"]))
    return pd.concat(parts).sort_index()


pit = walk_forward(U_PIT, "point-in-time")
fix = walk_forward(U_FIXED, "fixed 30")
print(f"Out-of-sample days: {sum(len(q['hold']) for q in pit):,} "
      f"({pd.Timestamp(pit[0]['hold'][0]).date()} to {pd.Timestamp(pit[-1]['hold'][-1]).date()})")

## 2. Ranking tools

Same as notebook 05, but the universe changes over time: each day, only that day's universe is ranked, and a stock that is delisted mid-quarter drops out after its final (delisting) return.

In [ ]:
def nw_t(x, lags=5):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; n = len(x); xc = x - x.mean()
    var = xc @ xc / n
    for k in range(1, lags + 1):
        var += 2 * (1 - k / (lags + 1)) * (xc[k:] @ xc[:-k]) / n
    return x.mean() / np.sqrt(var / n) if var > 0 else np.nan


def max_drawdown(daily):
    wealth = (1 + daily).cumprod()
    return (wealth / wealth.cummax() - 1).min()


def backtest(score, start, end, seed=1):
    s = score.loc[start:end]
    r = RET.reindex(index=s.index, columns=s.columns)
    valid = s.notna() & r.notna()
    s, r = s.where(valid), r.where(valid)
    e = r.sub(r.mean(axis=1), axis=0)
    ic = s.rank(axis=1).corrwith(e.rank(axis=1), axis=1).fillna(0.0)
    noise = np.random.default_rng(seed).uniform(0, 1e-9, s.shape)
    rk = (s.rank(axis=1) + noise).where(valid).rank(axis=1)
    n = valid.sum(axis=1)
    top = np.maximum(np.floor(n * TOP_FRAC), 1)
    w = (rk.gt(n - top, axis=0).astype(float).div(top, axis=0) - rk.le(top, axis=0).astype(float).div(top, axis=0)).where(valid, 0.0)
    gross = (w * r.fillna(0)).sum(axis=1)
    turnover = w.diff().abs().sum(axis=1)
    turnover.iloc[0] = w.iloc[0].abs().sum()
    net = gross - turnover * COST_BPS / 1e4
    return {"gross": gross, "net": net, "turnover": turnover, "ic": ic}


def summary(bt):
    g, n = bt["gross"], bt["net"]
    return pd.Series({"IC mean": bt["ic"].mean(), "IC t": nw_t(bt["ic"]), "LS bps/day": g.mean() * 1e4, "LS t": nw_t(g),
                      "Sharpe gross": g.mean() / g.std() * np.sqrt(252), "Sharpe net": n.mean() / n.std() * np.sqrt(252),
                      "turnover/day": bt["turnover"].mean(), "breakeven bps": g.mean() / bt["turnover"].mean() * 1e4,
                      "max drawdown": max_drawdown(g)})


def buy_hold(daily):
    return pd.Series({"LS bps/day": daily.mean() * 1e4, "LS t": nw_t(daily), "Sharpe gross": daily.mean() / daily.std() * np.sqrt(252),
                      "Sharpe net": daily.mean() / daily.std() * np.sqrt(252), "max drawdown": max_drawdown(daily)})
print("tools ready")

## 3. Choose λ on the 2006–2020 walk-forward

All 2006–2020 predictions are out of sample (each quarter's network only saw the past), so the λ with the best average rank IC over these years is a fair choice. One λ is chosen per universe.

In [ ]:
def ic_curve(results):
    return np.array([backtest(scores(results, k), None, TUNE_END)["ic"].mean() for k in range(len(ALPHAS))])

ic_pit, ic_fix = ic_curve(pit), ic_curve(fix)
k_pit, k_fix = int(np.argmax(ic_pit)), int(np.argmax(ic_fix))
edges_pit = [np.mean([(q["W"][:, :, k] != 0).sum() for q in pit]) for k in range(len(ALPHAS))]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.semilogx(ALPHAS, ic_pit, "o-", ms=3, label=f"point-in-time (best λ: {edges_pit[k_pit]:.0f} arrows per quarter on average)")
ax.semilogx(ALPHAS, ic_fix, "s-", ms=3, label="fixed 30 stocks")
ax.axvline(ALPHAS[k_pit], color="C0", ls="--"); ax.axvline(ALPHAS[k_fix], color="C1", ls=":")
ax.axhline(0, color="black", lw=0.8); ax.invert_xaxis()
ax.set_xlabel("λ  (left: few arrows, right: many arrows)"); ax.set_ylabel("mean daily rank IC, 2006–2020")
ax.set_title("Walk-forward rank IC along the λ grid"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"Point-in-time: best λ = {ALPHAS[k_pit]:.4f} (IC {ic_pit[k_pit]:.4f}); fixed 30: best λ = {ALPHAS[k_fix]:.4f} (IC {ic_fix[k_fix]:.4f})")

# λ by MSE (per stock): how many arrows does each stock keep?
arrows_mse = np.concatenate([[(q["W"][i, :, kk] != 0).sum() for i, kk in enumerate(q["k_mse"]) if kk >= 0] for q in pit])
print(f"λ by MSE (point-in-time): arrows per stock: median {np.median(arrows_mse):.0f}, "
      f"stock-quarters with NO arrows: {(arrows_mse == 0).mean():.0%} (their score is 0, so they sit in the middle of the ranking)")

## 4. Results: point-in-time universe

Signals: GDBN (one shared λ, chosen by rank IC), GDBN risk-adjusted (÷ predicted volatility), GDBN with λ by MSE (each stock's own λ, from the inner validation), reversal (minus yesterday's excess return), sentiment (yesterday's), random portfolios, and buy-and-hold comparisons (S&P 500 via SPY, and the universe equal-weighted).

In [ ]:
def signal_set(results, k):
    gd = scores(results, k)
    inuni = gd.notna()
    return {"GDBN": gd, "GDBN risk-adj": scores(results, k, "riskadj"), "GDBN (λ by MSE, per stock)": scores(results, kind="mse"),
            "Reversal": (-RET.shift(1)).reindex(index=gd.index, columns=gd.columns).where(inuni),
            "Sentiment": SENT.shift(1).reindex(index=gd.index, columns=gd.columns).where(inuni)}


def results_table(results, k, start, end):
    sig = signal_set(results, k)
    tab = pd.DataFrame({n: summary(backtest(s, start, end)) for n, s in sig.items()}).T
    gd = sig["GDBN"].loc[start:end]
    rand = pd.DataFrame([summary(backtest(pd.DataFrame(rng.normal(size=gd.shape), index=gd.index, columns=gd.columns).where(gd.notna()),
                                          start, end, seed=s)) for s in range(N_RANDOM)])
    tab.loc["Random (median)"] = rand.median()
    tab.loc["S&P 500 (SPY, buy & hold)"] = buy_hold(ETF.loc[gd.index, "SPY"])
    tab.loc["Universe, equal weight (buy & hold)"] = buy_hold(RET.reindex(index=gd.index, columns=gd.columns).where(gd.notna()).mean(axis=1))
    lo, hi = rand["Sharpe gross"].quantile([0.025, 0.975])
    return tab, (lo, hi)


tables = {}
for name, (start, end) in {"2006–2020 walk-forward": (None, TUNE_END), "2021–2022 validation": ("2021-01-01", EVAL_END)}.items():
    tab, (lo, hi) = results_table(pit, k_pit, start, end)
    tables[name] = tab
    print(f"===== {name} (point-in-time) =====   random 95% Sharpe range: [{lo:.2f}, {hi:.2f}]")
    print(tab.round(3).to_string(), "\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sig = signal_set(pit, k_pit)
for name in ["GDBN", "GDBN risk-adj", "GDBN (λ by MSE, per stock)", "Reversal", "Sentiment"]:
    bt = backtest(sig[name], None, EVAL_END)
    axes[0].plot((1 + bt["gross"]).cumprod(), label=name)
    axes[1].plot(bt["ic"].rolling(126).mean(), label=name)
for ax in axes:
    ax.axvline(pd.Timestamp("2021-01-01"), color="gray", ls="--", lw=1)
    ax.legend(fontsize=8)
axes[0].set_yscale("log"); axes[0].set_title("Long–short, growth of $1 (before costs; dashed line = validation starts)")
axes[1].axhline(0, color="black", lw=0.8); axes[1].set_title("Rank IC, 6-month rolling average")
plt.tight_layout(); plt.show()

## 5. Survivorship bias: point-in-time vs the fixed 30 stocks

Same model, same walk-forward, same rules. Only the universe differs.

- **Buy-and-hold rows:** owning the fixed 30 stocks (picked in 2026 because they are large **today**) vs owning the stocks that were actually largest **at the time**. The gap is pure hindsight.
- **GDBN rows:** how much of the ranking skill depends on the universe.

In [ ]:
rows = {}
for name, (start, end) in {"2006–2020": (None, TUNE_END), "2021–2022": ("2021-01-01", EVAL_END)}.items():
    for label, res, k in [("point-in-time", pit, k_pit), ("fixed 30 (hindsight)", fix, k_fix)]:
        gd = scores(res, k)
        s_g = summary(backtest(gd, start, end))
        s_r = summary(backtest(scores(res, k, "riskadj"), start, end))
        s_m = summary(backtest(scores(res, kind="mse"), start, end))
        ew = buy_hold(RET.reindex(index=gd.loc[start:end].index, columns=gd.columns).where(gd.loc[start:end].notna()).mean(axis=1))
        rows[(name, label)] = {"GDBN IC": s_g["IC mean"], "GDBN Sharpe gross": s_g["Sharpe gross"], "GDBN Sharpe net": s_g["Sharpe net"],
                               "risk-adj IC": s_r["IC mean"], "risk-adj Sharpe gross": s_r["Sharpe gross"],
                               "λ-by-MSE IC": s_m["IC mean"], "λ-by-MSE Sharpe gross": s_m["Sharpe gross"],
                               "equal-weight bps/day": ew["LS bps/day"], "equal-weight Sharpe": ew["Sharpe gross"]}
surv = pd.DataFrame(rows).T
surv

**Reading it:** if the fixed-30 equal-weight row earns much more than the point-in-time one, that is hindsight: those 30 were picked *because* they did well. If the GDBN's IC and Sharpe are similar in both universes, the ranking skill is **not** a survivorship artifact; if they drop a lot point-in-time, the earlier results were inflated.

## 6. The network over time (point-in-time, chosen λ)

Each quarter's network is a **snapshot** (the input format for the TBGNN). How does it change?
- **Arrows per quarter.**
- **Stock → stock return arrows within the same sector** (vs chance).
- **Persistence:** of the arrows between stocks present in two consecutive quarters, the share that appear in both (Jaccard overlap). Low = the network changes a lot from quarter to quarter.

In [ ]:
def edges_of(q, k):
    Wk = q["W"][:, :, k]
    i_idx, j_idx = np.nonzero(Wk)
    return pd.DataFrame({"quarter": q["r"], "target": [q["U"][i] for i in i_idx], "source": [q["cols"][j] for j in j_idx],
                         "weight": Wk[i_idx, j_idx]})

E = pd.concat([edges_of(q, k_pit) for q in pit], ignore_index=True)
E["src_kind"] = E["source"].str.split("|").str[1].fillna("etf")
E["src_permno"] = pd.to_numeric(E["source"].str.split("|").str[0], errors="coerce")
E["target_ticker"] = E["target"].map(TICKER)
E["source_ticker"] = E["src_permno"].map(TICKER).fillna(E["source"])

stats = []
prev = None
for r, g in E.groupby("quarter"):
    ss = g[(g["src_kind"] == "ret") & (g["src_permno"] != g["target"])]
    same = np.mean([SECTOR.get((r, a)) == SECTOR.get((r, b)) for a, b in zip(ss["src_permno"], ss["target"])]) if len(ss) else np.nan
    U = U_PIT[r]
    chance = np.mean([SECTOR.get((r, a)) == SECTOR.get((r, b)) for a in U for b in U if a != b])
    cur = set(zip(g["source"], g["target"]))
    jac = np.nan
    if prev is not None:
        both = set(U) & prev[1]
        both_str = {str(x) for x in both}
        keep = lambda e: e[1] in both and ("|" not in e[0] or e[0].split("|")[0] in both_str)   # both ends present in both quarters
        a, b = {e for e in cur if keep(e)}, {e for e in prev[0] if keep(e)}
        jac = len(a & b) / len(a | b) if (a | b) else np.nan
    stats.append({"quarter": r, "arrows": len(g), "stock->stock": len(ss), "same-sector share": same, "chance": chance, "persistence": jac})
    prev = (cur, set(U))
stats = pd.DataFrame(stats).set_index("quarter")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
stats["arrows"].plot(ax=axes[0], title="Arrows into the stocks, per quarter")
stats[["same-sector share", "chance"]].rolling(4).mean().plot(ax=axes[1], title="Stock→stock arrows within a sector (1-year avg)")
stats["persistence"].rolling(4).mean().plot(ax=axes[2], title="Arrows kept from the previous quarter (1-year avg)")
for ax in axes:
    ax.axvline(pd.Timestamp("2008-09-15"), color="red", ls=":", lw=1); ax.axvline(pd.Timestamp("2020-03-01"), color="red", ls=":", lw=1)
plt.tight_layout(); plt.show()
print(stats[["arrows", "stock->stock", "same-sector share", "chance", "persistence"]].describe().loc[["mean", "min", "max"]].round(3).to_string())
print("\nInputs used (share of arrows by type):", E["src_kind"].value_counts(normalize=True).round(3).to_dict())

## 7. Save results

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
for name, tab in tables.items():
    tab.to_csv(RESULTS / f"pit_results_{name.split()[0].replace('–', '_')}.csv")
surv.to_csv(RESULTS / "pit_vs_fixed_survivorship.csv")
E.to_csv(RESULTS / "pit_network_snapshots.csv", index=False)          # one network per quarter (TBGNN input format)
stats.to_csv(RESULTS / "pit_network_stats.csv")
pd.Series({"alpha_pit": ALPHAS[k_pit], "alpha_fixed": ALPHAS[k_fix], "sent_kind": SENT_KIND, "window": WINDOW}).to_csv(RESULTS / "pit_settings.csv")
print("Saved to", RESULTS)